In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_2.py ──
"""
Shared infrastructure for MLFP04 Exercise 2 — EM and Gaussian Mixture Models.

Contains: synthetic GMM data generation, Singapore e-commerce loader, scaler,
scoring helpers, and OUTPUT_DIR management.

Technique-specific code (the EM E-step / M-step from scratch, BIC/AIC sweeps,
covariance-type comparison, mixture-of-experts gating) does NOT live here.
It lives in the per-technique files under modules/mlfp04/solutions/ex_2/.
"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex2_gmm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 2D DATA — 3 well-separated Gaussians
# ════════════════════════════════════════════════════════════════════════
#
# Used to validate the from-scratch EM implementation. Three well-separated
# components make convergence obvious and let students verify the recovered
# parameters against the ground truth.

TRUE_MEANS: np.ndarray = np.array([[0.0, 0.0], [5.0, 2.0], [2.0, 6.0]])
TRUE_COVS: np.ndarray = np.array(
    [
        [[1.0, 0.3], [0.3, 0.8]],
        [[0.8, -0.2], [-0.2, 1.2]],
        [[1.5, 0.0], [0.0, 0.5]],
    ]
)
TRUE_WEIGHTS: np.ndarray = np.array([0.4, 0.35, 0.25])
N_SYNTH: int = 600


def make_synthetic_gmm(
    seed: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw N_SYNTH points from the 3-component GMM defined by TRUE_*.

    Returns (X, z_true) where z_true is the ground-truth component index.
    """
    rng = np.random.default_rng(seed)
    n_per = (TRUE_WEIGHTS * N_SYNTH).astype(int)
    n_per[-1] = N_SYNTH - n_per[:-1].sum()

    parts: list[np.ndarray] = []
    labels: list[int] = []
    for k, (mean, cov, n) in enumerate(zip(TRUE_MEANS, TRUE_COVS, n_per)):
        parts.append(rng.multivariate_normal(mean, cov, n))
        labels.extend([k] * n)

    X = np.vstack(parts)
    z = np.array(labels)

    # Shuffle so the order does not leak the label
    idx = rng.permutation(N_SYNTH)
    return X[idx], z[idx]


# ════════════════════════════════════════════════════════════════════════
# REAL DATA — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════
#
# We reuse the MLFP03 e-commerce customer dataset (~6K rows, Singapore)
# for every real-data task in this exercise. Segmentation is the business
# frame: the GMM will propose soft customer segments that marketing can
# score on expected value.


def load_customers_scaled() -> (
    tuple[np.ndarray, pl.DataFrame, list[str], StandardScaler]
):
    """Return (X_scaled, customers_df, feature_cols, scaler).

    The DataFrame and feature_cols are returned so technique files can
    join cluster labels back onto the original rows for segment profiling.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    customers = customers.drop_nulls(subset=feature_cols)
    X, _, _ = to_sklearn_input(customers, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, customers, feature_cols, scaler


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING — for BIC/AIC interpretation
# ════════════════════════════════════════════════════════════════════════


def count_gmm_params(n_components: int, n_features: int, cov_type: str) -> int:
    """Number of free parameters in a GMM given components, features, cov type.

    Used to explain the BIC/AIC ranking across covariance types.
    """
    d = n_features
    k = n_components
    if cov_type == "full":
        return k * (d * (d + 1) // 2 + d + 1) - 1
    if cov_type == "tied":
        return d * (d + 1) // 2 + k * (d + 1) - 1
    if cov_type == "diag":
        return k * (2 * d + 1) - 1
    if cov_type == "spherical":
        return k * (d + 2) - 1
    raise ValueError(f"Unknown cov_type: {cov_type}")


# ════════════════════════════════════════════════════════════════════════
# SCORING HELPERS
# ════════════════════════════════════════════════════════════════════════


def safe_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    """Silhouette with a graceful fallback when only one cluster is present."""
    if len(set(labels.tolist())) < 2:
        return float("nan")
    return float(silhouette_score(X, labels))


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 2.2: sklearn GMM and BIC/AIC Model Selection
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Fit a GaussianMixture through kailash-ml's sklearn bridge
#   - Verify the library result matches the from-scratch EM from 2.1
#   - Sweep K using BIC and AIC to select the number of components
#   - Explain WHY BIC is more conservative than AIC (different penalties)
#   - Read a BIC/AIC curve and recognise the elbow
#
# PREREQUISITES: 01_em_from_scratch.py
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — BIC and AIC as log-likelihood minus complexity penalty
#   2. Build — fit_sklearn_gmm helper + bic_aic_sweep
#   3. Train — fit K=2..8 on Singapore e-commerce customers
#   4. Visualise — BIC/AIC curves + silhouette overlay
#   5. Apply — Shopee SEA customer segmentation at enterprise scale
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.mixture import GaussianMixture

from kailash_ml import ModelVisualizer



## THEORY — BIC and AIC


In [ ]:
#   BIC = k * log(n) - 2 * log_lik   (Schwarz)
#   AIC = 2 * k      - 2 * log_lik   (Akaike)
#
# Lower is better for both. BIC's log(n) penalty grows with sample size,
# making it more conservative than AIC for large n. BIC is consistent
# (picks the true K as n -> infinity); AIC is efficient (minimises
# prediction error but may over-parameterise).
#
# PRACTICAL RULE: when BIC and AIC disagree, prefer BIC.



## TASK 2 — BUILD: fit_sklearn_gmm + bic_aic_sweep


Fit a sklearn GaussianMixture and return the fitted estimator.


Fit a GMM for each K and record BIC, AIC, log-lik, silhouette.


In [ ]:
def fit_sklearn_gmm(
    X: np.ndarray,
    n_components: int,
    cov_type: str = "full",
    random_state: int = 42,
) -> GaussianMixture:
    # TODO: construct GaussianMixture with n_components, covariance_type=cov_type,
    # random_state=random_state, max_iter=200 — then .fit(X) and return it.
    gmm = ____
    ____
    return gmm


def bic_aic_sweep(
    X: np.ndarray,
    k_range: range,
    cov_type: str = "full",
) -> dict[int, dict[str, float]]:
    results: dict[int, dict[str, float]] = {}
    for k in k_range:
        gmm = fit_sklearn_gmm(X, k, cov_type=cov_type)
        labels = gmm.predict(X)
        # TODO: fill the results dict with bic, aic, log_lik, silhouette
        # Hint: gmm.bic(X), gmm.aic(X), gmm.score(X) * X.shape[0], safe_silhouette(X, labels)
        results[k] = {
            "bic": ____,
            "aic": ____,
            "log_lik": ____,
            "silhouette": ____,
        }
    return results



## TASK 3 — TRAIN: fit K=2..8 on Singapore e-commerce customers


In [ ]:
X_scaled, customers, feature_cols, _ = load_customers_scaled()

print("=" * 70)
print("  Singapore e-commerce customers")
print("=" * 70)
print(f"Rows: {X_scaled.shape[0]}  Features: {X_scaled.shape[1]}")

print("\nSweeping K = 2..8 with sklearn GaussianMixture (cov_type=full)...")
# TODO: call bic_aic_sweep(X_scaled, range(2, 9))
sweep = ____

print(f"\n{'K':>4} {'BIC':>12} {'AIC':>12} {'log_lik':>12} {'silhouette':>12}")
print("─" * 56)
for k, v in sweep.items():
    print(
        f"{k:>4} {v['bic']:>12.0f} {v['aic']:>12.0f} "
        f"{v['log_lik']:>12.0f} {v['silhouette']:>12.4f}"
    )

# TODO: pick K that minimises BIC, K that minimises AIC, K that maximises silhouette
# Hint: min(sweep.items(), key=lambda kv: kv[1]["bic"])[0]
best_k_bic = ____
best_k_aic = ____
best_k_sil = ____

print(f"\nBest K by BIC:        {best_k_bic}")
print(f"Best K by AIC:        {best_k_aic}")
print(f"Best K by silhouette: {best_k_sil}")



### Checkpoint 1


In [ ]:
assert best_k_bic in range(2, 9), "BIC-optimal K must be in the searched range"
assert best_k_aic in range(2, 9), "AIC-optimal K must be in the searched range"
bic_values = [v["bic"] for v in sweep.values()]
assert max(bic_values) - min(bic_values) > 1.0, "BIC should vary across K"
print("\n[ok] Checkpoint 1 passed — BIC/AIC sweep produced a usable ranking")



## TASK 4 — VISUALISE: BIC/AIC curves + silhouette overlay


In [ ]:
viz = ModelVisualizer()
comparison = {
    f"K={k}": {"BIC": v["bic"], "AIC": v["aic"], "silhouette": v["silhouette"]}
    for k, v in sweep.items()
}
# TODO: call viz.metric_comparison(comparison)
fig = ____
fig.update_layout(title="GMM model selection: BIC, AIC, silhouette vs K")
fig.write_html(str(out_path("ex2_sklearn_bic_aic.html")))
print(f"\nSaved: {out_path('ex2_sklearn_bic_aic.html')}")



### Checkpoint 2


In [ ]:
assert out_path("ex2_sklearn_bic_aic.html").exists(), "BIC/AIC plot must be written"
print("[ok] Checkpoint 2 passed — BIC/AIC visualisation written")



## TASK 5 — APPLY: Shopee SEA customer segmentation at scale


In [ ]:
# SCENARIO: Shopee (Sea Group, Singapore-HQ) runs the largest e-commerce
# platform in SEA — ~80M monthly actives across 7 countries. The growth
# team segments customers for lifecycle campaigns: welcome, re-engagement,
# win-back, retention.
#
# Why BIC over "pick K=4 because marketing likes round numbers":
#   When true structure has 6-7 segments, K=4 merges genuinely different
#   buyers and creative targeted to the merged segment misfires on half
#   the members.
#
# BUSINESS IMPACT (from a Lazada 2024 study on SEA marketplaces):
#   - Monthly campaign spend: ~S$420,000
#   - Baseline open rate with K=4: ~18%
#   - BIC-selected K=6 lifted campaign engagement by ~23%
#   - 23% uplift on S$420K already-paid spend = S$97K/month = S$1.16M/year
#     in recovered ROI, zero extra infra cost.

# TODO: fit a GMM at best_k_bic and extract its mixing weights
best_gmm = ____
segment_weights = best_gmm.weights_

print("\n" + "=" * 70)
print(f"  APPLY — Shopee SEA segmentation (BIC-optimal K={best_k_bic})")
print("=" * 70)
for k, w in enumerate(segment_weights):
    print(f"  Segment {k}: weight={w:.3f}  (~{w * 100:.1f}% of customers)")



### Checkpoint 3


In [ ]:
assert len(segment_weights) == best_k_bic, "weights length must match K"
assert abs(segment_weights.sum() - 1.0) < 1e-6, "weights must sum to 1"
print("\n[ok] Checkpoint 3 passed — BIC-optimal segmentation produced")



## REFLECTION


[x] Fit sklearn GaussianMixture as a drop-in for your from-scratch EM
  [x] Compute BIC and AIC to penalise model complexity
  [x] Select K by the BIC elbow (here: K={best_k_bic})
  [x] Shopee SEA scenario: BIC-guided K turns into S$1.16M/year in ROI

  Next: 03_covariance_types.py — same K, four cluster shapes, BIC picks.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

